In [ ]:

# ============================================================
# DAY 2 AI AGENTS WORKSHOP
# PROJECT: SMART GROCERY SHOPPING ASSISTANT
# ============================================================

# 1. INSTALLATION
!pip install -q openai

# 2. IMPORT LIBRARIES AND API CONNECTION

from google.colab import userdata
from openai import OpenAI, RateLimitError, APIConnectionError
import json
import functools

client = OpenAI(
    api_key=userdata.get("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

MODEL = "openai/gpt-oss-20b"

print("Groq API connected successfully!")


# 3. GROCERY PRODUCT CATALOGUE

PRODUCTS = {
    "rice": {"price": 60, "unit": "kg"},
    "sugar": {"price": 45, "unit": "kg"},
    "milk": {"price": 30, "unit": "litre"},
    "bread": {"price": 40, "unit": "packet"},
    "eggs": {"price": 7, "unit": "piece"},
    "apples": {"price": 120, "unit": "kg"},
    "tomatoes": {"price": 35, "unit": "kg"},
    "potatoes": {"price": 30, "unit": "kg"},
    "oil": {"price": 150, "unit": "litre"}
}

print("\nAvailable Grocery Products:")
for item, details in PRODUCTS.items():
    print(f"{item.title()}: Rs.{details['price']} per {details['unit']}")


# 4. SYSTEM PROMPT

SYSTEM = f"""
ROLE:
You are a friendly Smart Grocery Shopping Assistant.

FACTS:
Grocery product catalogue with prices in rupees:
{json.dumps(PRODUCTS)}

RULES:
1. Only provide prices available in the catalogue.
2. Never invent product prices or availability.
3. If a product is not listed, say "I'll check with the store."
4. Help customers calculate grocery expenses.
5. Remember the customer's name and shopping requests
   during the conversation.
6. Never claim that an order has been placed or paid for.
7. Help customers check whether their shopping fits
   within their budget.

FORMAT:
Keep responses under 50 words.
Be friendly and helpful.
Reply in English or Telugu, matching the customer.
"""


# 5. DECORATOR

def log_call(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        result = func(*args, **kwargs)
        print(f"[LOG] {func.__name__}{args} -> {result!r}")
        return result
    return wrapper


# 6. PRICE CALCULATION FUNCTION

@log_call
def get_price(item: str, qty: int = 1) -> int:
    """
    Calculate the total price of a grocery product.

    Args:
        item: Name of the grocery product.
        qty: Quantity required.

    Returns:
        Total price in rupees.
    """

    item = item.lower().strip()

    if item not in PRODUCTS:
        raise ValueError("Product not available.")

    if qty <= 0:
        raise ValueError("Quantity must be greater than zero.")

    return PRODUCTS[item]["price"] * qty


print("\nPrice for 3 kg rice:", get_price("rice", 3))
print("Price for 2 litres milk:", get_price("milk", 2))


# 7. CHAT HISTORY AND MEMORY

history = [
    {"role": "system", "content": SYSTEM}
]

MAX_TURNS = 8


# 8. MAIN CHAT FUNCTION

def chat(user_text: str) -> str:

    history.append({
        "role": "user",
        "content": user_text
    })

    recent = [history[0]] + history[1:][-MAX_TURNS:]

    try:
        response = client.chat.completions.create(
            model=MODEL,
            messages=recent,
            temperature=0.3
        )

        reply = response.choices[0].message.content

    except RateLimitError:
        reply = "The service is busy. Please try again later."

    except APIConnectionError:
        reply = "Unable to connect to the server. Please try again."

    except Exception as e:
        reply = f"An error occurred: {str(e)}"

    history.append({
        "role": "assistant",
        "content": reply
    })

    return reply


# ============================================================
# ACTIVITY 1: SYSTEM PROMPT EXPERIMENT
# ============================================================

print("\n========== ACTIVITY 1: PROMPT EXPERIMENT ==========")

prompt_without_rules = f"""
ROLE: You are a grocery shopping assistant.
FACTS: {json.dumps(PRODUCTS)}
"""

question = "Do you have chocolate available?"

try:
    result1 = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": prompt_without_rules},
            {"role": "user", "content": question}
        ],
        temperature=0.3
    )

    print("Without Rules:")
    print(result1.choices[0].message.content)

    result2 = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": SYSTEM},
            {"role": "user", "content": question}
        ],
        temperature=0.3
    )

    print("\nWith Rules:")
    print(result2.choices[0].message.content)

except Exception as e:
    print("Prompt experiment error:", e)


# ============================================================
# ACTIVITY 2: FEW-SHOT CLASSIFICATION
# ============================================================

print("\n========== ACTIVITY 2: FEW-SHOT CLASSIFIER ==========")

def classify_message(message):

    prompt = """
You are a grocery message classifier.

Classify the message into one category:
PRODUCT_QUERY, PRICE_QUERY, SHOPPING_LIST, OTHER.

Examples:

Message: Do you have milk?
Category: PRODUCT_QUERY

Message: What is the price of rice?
Category: PRICE_QUERY

Message: Add apples to my list.
Category: SHOPPING_LIST

Message: Tell me a joke.
Category: OTHER

Return only the category name.

Message: """ + message

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature=0
    )

    return response.choices[0].message.content.strip()


test_messages = [
    "Do you sell eggs?",
    "What is the cost of milk?",
    "Add rice and potatoes to my list.",
    "Tell me a joke.",
    "Is bread available?"
]

for message in test_messages:
    try:
        print("\nMessage:", message)
        print("Category:", classify_message(message))
    except Exception as e:
        print("Classification error:", e)


# ============================================================
# ACTIVITY 3: JSON ORDER EXTRACTION
# ============================================================

print("\n========== ACTIVITY 3: JSON EXTRACTION ==========")

order_message = """
My name is Anusha. I want 2 kg rice,
1 litre milk and 1 packet bread.
My budget is 200 rupees.
"""

json_prompt = """
Extract the grocery order details.

Return ONLY valid JSON with:
name, items, budget.

The items field must contain objects with:
item, quantity, unit.

Use null for missing values.
Do not include Markdown or explanations.

Customer message:
""" + order_message

try:
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "user", "content": json_prompt}
        ],
        temperature=0
    )

    json_output = response.choices[0].message.content

    # Remove Markdown fences if the model adds them
    cleaned_output = json_output.strip()

    if cleaned_output.startswith("```"):
        cleaned_output = cleaned_output.split("\n", 1)[1]
        cleaned_output = cleaned_output.rsplit("```", 1)[0].strip()

    order_data = json.loads(cleaned_output)

    print("Extracted Order:")
    print(json.dumps(order_data, indent=4))

    print("\nCustomer:", order_data["name"])
    print("Budget:", order_data["budget"])

    for item in order_data["items"]:
        print(item)

except (json.JSONDecodeError, TypeError, KeyError) as e:
    print("Unable to process JSON:", e)

except Exception as e:
    print("JSON extraction error:", e)


# ============================================================
# ACTIVITY 4: MEMORY EXPERIMENT
# ============================================================

print("\n========== ACTIVITY 4: MEMORY EXPERIMENT ==========")

MAX_TURNS = 2

history = [
    {"role": "system", "content": SYSTEM}
]

memory_messages = [
    "My name is Anusha.",
    "I like buying fresh fruits.",
    "What is the price of apples?"
]

for message in memory_messages:
    print("\nYou:", message)
    print("Bot:", chat(message))

question = "What is my name?"

print("\nYou:", question)
print("Bot:", chat(question))

print("\nObservation:")
print("With a smaller memory limit, older messages may be removed.")
print("Therefore, the chatbot may forget earlier information.")


# ============================================================
# ACTIVITY 5: OWN USE CASE IMPLEMENTATION
# ============================================================

print("\n========== ACTIVITY 5: OUR OWN USE CASE ==========")

MY_CATALOGUE = {
    "rice": 60,
    "sugar": 45,
    "milk": 30,
    "bread": 40,
    "eggs": 7,
    "apples": 120,
    "tomatoes": 35,
    "potatoes": 30,
    "oil": 150
}


def calculate_grocery_total(item: str, quantity: int = 1) -> int:
    """
    Calculate the total cost of a grocery product.
    """

    item = item.lower().strip()

    if item not in MY_CATALOGUE:
        raise ValueError("Product not found.")

    if quantity <= 0:
        raise ValueError("Quantity must be positive.")

    return MY_CATALOGUE[item] * quantity


MY_SYSTEM = f"""
ROLE:
You are a Smart Grocery Shopping Assistant.

FACTS:
Product catalogue:
{json.dumps(MY_CATALOGUE)}

RULES:
Only provide prices from the catalogue.
Never invent product details.
Help customers calculate grocery expenses.
If a product is missing, say you will check with the store.

FORMAT:
Give short, friendly and helpful responses.
Use simple English or Telugu.
"""

print("Our grocery catalogue:", MY_CATALOGUE)
print("Cost of 3 kg rice:", calculate_grocery_total("rice", 3))
print("Our custom system prompt created successfully!")


# ============================================================
# FINAL CHATBOT: INTERACTIVE CONVERSATION
# ============================================================

print("\n==============================================")
print("       SMART GROCERY SHOPPING ASSISTANT")
print("==============================================")
print("Hello! Welcome to our Grocery Shopping Assistant.")
print("Ask about products, prices or your shopping list.")
print("Type 'exit' or 'bye' to stop.\n")

# Restore the original memory settings
MAX_TURNS = 8

history = [
    {"role": "system", "content": SYSTEM}
]

while True:

    user_input = input("You: ").strip()

    if user_input.lower() in ["bye", "exit", "quit"]:
        print("Bot: Thank you for using Smart Grocery Shopping Assistant!")
        break

    if not user_input:
        continue

    print("Bot:", chat(user_input))

# ============================================================
# END OF DAY 2 PROJECT
# ============================================================


Groq API connected successfully!

Available Grocery Products:
Rice: Rs.60 per kg
Sugar: Rs.45 per kg
Milk: Rs.30 per litre
Bread: Rs.40 per packet
Eggs: Rs.7 per piece
Apples: Rs.120 per kg
Tomatoes: Rs.35 per kg
Potatoes: Rs.30 per kg
Oil: Rs.150 per litre
[LOG] get_price('rice', 3) -> 180

Price for 3 kg rice: 180
[LOG] get_price('milk', 2) -> 60
Price for 2 litres milk: 60

========== ACTIVITY 1: PROMPT EXPERIMENT ==========
Without Rules:
I’m afraid we don’t have chocolate in stock right now. Is there anything else you’re looking for?

With Rules:
I’ll check with the store.

========== ACTIVITY 2: FEW-SHOT CLASSIFIER ==========

Message: Do you sell eggs?
Category: PRODUCT_QUERY

Message: What is the cost of milk?
Category: PRICE_QUERY

Message: Add rice and potatoes to my list.
Category: SHOPPING_LIST

Message: Tell me a joke.
Category: OTHER

Message: Is bread available?
Category: PRODUCT_QUERY

========== ACTIVITY 3: JSON EXTRACTION ==========
Extracted Order:
{
    "name": "Anu